# load_openalex_work

Prototipo del nodo `load_openalex_work` del pipeline `load_openalex`. No guarda datasets.


In [ ]:
import pandas as pd
from pandas import json_normalize

%load_ext kedro.ipython


In [ ]:
df_work_raw = catalog.load('raw/openalex/work/parquet/work_dev')
df_work_raw.head(2)


In [ ]:
def _select_with_metadata(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    df = _add_openalex_extracted_metadata(df)
    return df.loc[:, [*columns, *_EXTRACTED_META_COLS]].copy()


In [ ]:
def _stringify_object_columns(
    df: pd.DataFrame,
    exclude_columns: list[str] | None = None,
) -> pd.DataFrame:
    exclude_columns = set(exclude_columns or [])
    for column in df.columns:
        if column in exclude_columns:
            continue
        if pd.api.types.is_object_dtype(df[column]):
            df[column] = df[column].where(df[column].notna(), pd.NA).astype("string")
    return df


In [ ]:
def _serialize_nested_value(value):
    if value is None or value is pd.NA:
        return value
    if hasattr(value, "tolist") and not isinstance(value, (str, bytes)):
        value = value.tolist()
    if isinstance(value, (dict, list, tuple, set)):
        return json.dumps(value, ensure_ascii=False, default=str)
    return value


In [ ]:
def _serialize_nested_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for column in df.columns:
        if pd.api.types.is_object_dtype(df[column]):
            df[column] = df[column].map(_serialize_nested_value)
    return df


In [ ]:
def _add_openalex_extracted_metadata(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    for col in _EXTRACTED_META_COLS:
        if col not in df.columns:
            df[col] = pd.NA
    df["extract_datetime"] = pd.to_datetime(df["extract_datetime"], errors="coerce")
    df["_extract_datetime"] = pd.to_datetime(df["_extract_datetime"], errors="coerce")
    if "extract_date" in df.columns:
        df["extract_date"] = pd.to_datetime(df["extract_date"], errors="coerce").dt.date
    return df


In [ ]:
def _add_openalex_loaded_metadata(df: pd.DataFrame, load_datetime=None) -> pd.DataFrame:
    df = _serialize_nested_columns(df)
    if load_datetime is None:
        load_datetime = pd.Timestamp.now(tz="UTC").floor("s").tz_localize(None)
    load_datetime = pd.to_datetime(load_datetime)
    df["_load_datetime"] = load_datetime
    return df


In [ ]:
def load_openalex_work(df_work_raw, load_datetime=None):
    """Limpia y transforma los datos de OpenAlex para su almacenamiento en una base de datos relacional."""
    df_work_raw = _add_openalex_extracted_metadata(df_work_raw)

    expected_columns = [
        'id',
        # 'doi', # doi existe en ids
        'title',
        'display_name',
        'publication_year',
        'publication_date',
        'ids',
        'language',
        'primary_location',
        'type',
        'type_crossref',
        # 'indexed_in',
        'open_access',
        # 'authorships',
        # 'institution_assertions',
        'countries_distinct_count',
        'institutions_distinct_count',
        # 'corresponding_author_ids',
        # 'corresponding_institution_ids',
        'apc_list',
        'apc_paid',
        'fwci',
        'has_fulltext',
        'fulltext_origin',
        'cited_by_count',
        'citation_normalized_percentile',
        'cited_by_percentile_year',
        'biblio',
        'is_retracted',
        'is_paratext',
        'primary_topic',
        # 'topics',
        # 'keywords',
        # 'concepts',
        # 'mesh',
        'locations_count',
        # 'locations',
        'best_oa_location',
        # 'sustainable_development_goals',
        # 'grants',
        # 'datasets',
        # 'versions',
        'referenced_works_count',
        # 'referenced_works',
        # 'related_works',
        # 'abstract_inverted_index',
        # 'abstract_inverted_index_v3',
        'cited_by_api_url',
        # 'counts_by_year',
        'updated_date',
        'created_date',
        *_CORE_EXTRACTED_META_COLS,
        '_filter_param',
        '_filter_value',
        '_extract_datetime',
    ]

    df_work = df_work_raw.reindex(columns=expected_columns).reset_index(drop=True).copy()

    # Agregar columnas faltantes con NaN
    for col in expected_columns:
        if col not in df_work.columns:
            df_work[col] = pd.NA

    # ids
    df_ids = pd.json_normalize(df_work['ids']).reset_index(drop=True)
    df_work = pd.concat([df_work, df_ids], axis=1)
    df_work.drop(columns=['ids'], inplace=True)    

    # primary_location
    df_primary_location = pd.json_normalize(df_work['primary_location']).reset_index(drop=True)
    df_primary_location.rename(columns=lambda col: f'primary_location.{col}', inplace=True)
    df_primary_location.drop(columns=[
        'primary_location.source',
        'primary_location.source.issn',
        'primary_location.source.host_organization_lineage',
        'primary_location.source.host_organization_lineage_names'
        ],
        inplace=True,
        errors='ignore')

    df_work = pd.concat([df_work, df_primary_location], axis=1)
    df_work.drop(columns=['primary_location'], inplace=True)    

    # openacess
    df_openaccess_expanded = pd.json_normalize(df_work['open_access'])
    df_work = pd.concat([df_work, df_openaccess_expanded], axis=1)
    df_work.drop(columns=['open_access'], inplace=True)    

    # apc_list
    df_apc_list = pd.json_normalize(df_work['apc_list'])
    df_apc_list.rename(columns=lambda col: f'apc_list.{col}', inplace=True)
    df_work = pd.concat([df_work, df_apc_list], axis=1)
    df_work.drop(columns=['apc_list'], inplace=True)    
 
    # apc_paid
    df_apc_paid = pd.json_normalize(df_work['apc_paid'])
    df_apc_paid.rename(columns=lambda col: f'apc_paid.{col}', inplace=True)
    df_work = pd.concat([df_work, df_apc_paid], axis=1)
    df_work.drop(columns=['apc_paid'], inplace=True)    
 
    # citation_normalized_percentile
    df_citation_normalized_percentile = pd.json_normalize(df_work['citation_normalized_percentile'])
    df_citation_normalized_percentile.rename(columns=lambda col: f'citation_normalized_percentile.{col}', inplace=True)
    df_work = pd.concat([df_work, df_citation_normalized_percentile], axis=1)
    df_work.drop(columns=['citation_normalized_percentile'], inplace=True)    

    # cited_by_percentile_year
    df_cited_by_percentile_year = pd.json_normalize(df_work['cited_by_percentile_year'])
    df_cited_by_percentile_year.rename(columns=lambda col: f'cited_by_percentile_year.{col}', inplace=True)
    df_work = pd.concat([df_work, df_cited_by_percentile_year], axis=1)
    df_work.drop(columns=['cited_by_percentile_year'], inplace=True)    

    # biblio
    df_biblio = pd.json_normalize(df_work['biblio'])
    df_biblio.rename(columns=lambda col: f'biblio.{col}', inplace=True)
    df_work = pd.concat([df_work, df_biblio], axis=1)
    df_work.drop(columns=['biblio'], inplace=True)

    # primary_topic
    df_primary_topic = pd.json_normalize(df_work['primary_topic'])
    df_primary_topic.rename(columns=lambda col: f'primary_topic.{col}', inplace=True)
    df_work = pd.concat([df_work, df_primary_topic], axis=1)
    df_work.drop(columns=['primary_topic'], inplace=True)    

    # best_oa_location
    df_best_oa_location = pd.json_normalize(df_work['best_oa_location'])
    df_best_oa_location.rename(columns=lambda col: f'best_oa_location.{col}', inplace=True)
    df_best_oa_location.drop(columns=[
        'best_oa_location.source.host_organization_lineage',
        'best_oa_location.source.host_organization_lineage_names',
        'best_oa_location.source.issn',
    ], inplace=True, errors='ignore')
    
    df_work = pd.concat([df_work, df_best_oa_location], axis=1)
    df_work.drop(columns=['best_oa_location'], inplace=True)    

    df_work = _add_openalex_loaded_metadata(df_work, load_datetime=load_datetime)

    # Convertir tipos de datos automáticamente
    df_work = df_work.convert_dtypes()

    return df_work


In [ ]:
df_work = load_openalex_work(df_work_raw)


In [ ]:
pd.DataFrame([{'dataset': 'df_work', 'rows': len(df_work), 'columns': len(df_work.columns)}])


In [ ]:
df_work.head(2)
